# Import necessary libraries

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

import zipfile
import os

import nltk
import re
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Extract the zip file

In [38]:
zip_path = '/content/amazon_co-ecommerce_sample.csv.zip'  # Path to your ZIP file
extract_path = '/content'  # Destination folder

# Create extraction directory if not exists
os.makedirs(extract_path, exist_ok=True)

# Extract the ZIP file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Files extracted to: {extract_path}")

Files extracted to: /content


In [39]:
amazon_df = pd.read_csv('/content/amazon_co-ecommerce_sample.csv')

# Visualize the dataset and perform necessary cleaning

Removing uncessary columns from the dataset that will not contribute much to the recommender model and removing rows of data that has null values as out dataset is large comprising of 10k rows, even if we lose some of the rows it won't affect much to our recommender and as it will be much better to do this than adding them.


One thing that we can also do is seperating those null valued rows and cluster them as test data and try to fill out those NaN columns in those with our recommender model.

In [43]:
amazon_df.head(2)

,uniq_id,product_name,manufacturer,price,number_available_in_stock,number_of_reviews,number_of_answered_questions,average_review_rating,amazon_category_and_sub_category,customers_who_bought_this_item_also_bought,description,product_information,product_description,items_customers_buy_after_viewing_this_item,customer_questions_and_answers,customer_reviews,sellers
0,eac7efa5dbd3d667f26eb3d3ab504464,Hornby 2014 Catalogue,Hornby,£3.42,5 new,15,1.0,4.9 out of 5 stars,Hobbies > Model Trains & Railway Sets > Rail V...,http://www.amazon.co.uk/Hornby-R8150-Catalogue...,Product Description Hornby 2014 Catalogue Box ...,Technical Details Item Weight640 g Product Dim...,Product Description Hornby 2014 Catalogue Box ...,http://www.amazon.co.uk/Hornby-R8150-Catalogue...,Does this catalogue detail all the previous Ho...,Worth Buying For The Pictures Alone (As Ever) ...,"{""seller""=>[{""Seller_name_1""=>""Amazon.co.uk"", ..."
1,b17540ef7e86e461d37f3ae58b7b72ac,FunkyBuys® Large Christmas Holiday Express Fes...,FunkyBuys,£16.99,NaN,2,1.0,4.5 out of 5 stars,Hobbies > Model Trains & Railway Sets > Rail V...,http://www.amazon.co.uk/Christmas-Holiday-Expr...,Size Name:Large FunkyBuys® Large Christmas Hol...,Technical Details Manufacturer recommended age...,Size Name:Large FunkyBuys® Large Christmas Hol...,http://www.amazon.co.uk/Christmas-Holiday-Expr...,can you turn off sounds // hi no you cant turn...,Four Stars // 4.0 // 18 Dec. 2015 // By\n \...,"{""seller""=>{""Seller_name_1""=>""UHD WHOLESALE"", ..."


In [48]:
print(amazon_df['product_information'][0])
print(amazon_df['product_description'][1])

Technical Details Item Weight640 g Product Dimensions29.6 x 20.8 x 1 cm Manufacturer recommended age:6 years and up Item model numberR8148 Main Language(s)English manual, English Number of Game Players1 Number of Puzzle Pieces1 Assembly RequiredNo Scale1:72 Engine Typeelectric Track Width/GaugeHO Batteries Required?No Batteries Included?No Material Type(s)Paper Material Care InstructionsNo Remote Control Included?No Radio Control Suitabilityindoor Colorwhite    Additional Information ASINB00HJ208KO Best Sellers Rank 52,854 in Toys & Games (See top 100) #69 in Toys & Games > Model Trains & Railway Sets > Rail Vehicles > Trains Shipping Weight640 g Delivery Destinations:Visit the Delivery Destinations Help page to see where this item can be delivered. Date First Available24 Dec. 2013    Feedback  Would you like to update product info or give feedback on images?
Size Name:Large FunkyBuys® Large Christmas Holiday Express Festive Train Set (SI-TY1017) Toy Light / Sounds / Battery Operated &

In [41]:
amazon_df.columns

Index(['uniq_id', 'product_name', 'manufacturer', 'price',
       'number_available_in_stock', 'number_of_reviews',
       'number_of_answered_questions', 'average_review_rating',
       'amazon_category_and_sub_category',
       'customers_who_bought_this_item_also_bought', 'description',
       'product_information', 'product_description',
       'items_customers_buy_after_viewing_this_item',
       'customer_questions_and_answers', 'customer_reviews', 'sellers'],
      dtype='object')

In [49]:
amazon_df.drop(columns = ['uniq_id', 'manufacturer', 'price',
                          'number_available_in_stock', 'number_of_reviews',
                          'number_of_answered_questions', 'average_review_rating',
                          'customers_who_bought_this_item_also_bought', 'product_information', 'product_description',
                          'items_customers_buy_after_viewing_this_item',
                          'customer_questions_and_answers', 'customer_reviews', 'sellers'], axis=1, inplace = True)

In [60]:
amazon_df.head()

,product_name,amazon_category_and_sub_category,description
0,Hornby 2014 Catalogue,Hobbies > Model Trains & Railway Sets > Rail V...,Product Description Hornby 2014 Catalogue Box ...
1,FunkyBuys® Large Christmas Holiday Express Fes...,Hobbies > Model Trains & Railway Sets > Rail V...,Size Name:Large FunkyBuys® Large Christmas Hol...
2,CLASSIC TOY TRAIN SET TRACK CARRIAGES LIGHT EN...,Hobbies > Model Trains & Railway Sets > Rail V...,BIG CLASSIC TOY TRAIN SET TRACK CARRIAGE LIGHT...
3,HORNBY Coach R4410A BR Hawksworth Corridor 3rd,Hobbies > Model Trains & Railway Sets > Rail V...,Hornby 00 Gauge BR Hawksworth 3rd Class W 2107...
4,Hornby 00 Gauge 0-4-0 Gildenlow Salt Co. Steam...,Hobbies > Model Trains & Railway Sets > Rail V...,Product Description Hornby RailRoad 0-4-0 Gild...


In [51]:
amazon_df.shape

(10000, 3)

In [52]:
amazon_df.isnull().sum()

,0
product_name,0
amazon_category_and_sub_category,690
description,651


In [53]:
amazon_df.dropna(inplace = True)

In [54]:
amazon_df.isnull().sum()

,0
product_name,0
amazon_category_and_sub_category,0
description,0


In [58]:
amazon_df.shape

(8692, 3)

# Define a function that tokenizes and stems the text of the product titles, category and descriptions using the NLTK library.

Overall, this function is useful for preprocessing text data for tasks such as natural language processing and machine learning, where reducing the number of unique words in the text can improve model performance. Stemming can help by reducing each word to its base form, allowing the model to treat variations of the same word (e.g., "run", "running", "ran") as equivalent.

In [56]:
ps = PorterStemmer()

def tokenize_stem(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove digits and special characters except whitespace
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize the text word-wise
    tokens = nltk.word_tokenize(text)

    # Keep only base form of words
    stemmed_tokens = [ps.stem(token) for token in tokens]

    # Return words joined by space
    return " ".join(stemmed_tokens)

In [62]:
amazon_df['stemmed_tokens'] = amazon_df.apply(lambda row: tokenize_stem(row['product_name'] + " " + row['amazon_category_and_sub_category'] + " " + row['description']), axis = 1)

In [63]:
amazon_df['stemmed_tokens'].head()

,stemmed_tokens
0,hornbi catalogu hobbi model train railway set ...
1,funkybuy larg christma holiday express festiv ...
2,classic toy train set track carriag light engi...
3,hornbi coach ra br hawksworth corridor rd hobb...
4,hornbi gaug gildenlow salt co steam locomot mo...


In [79]:
amazon_df.head()

,product_name,amazon_category_and_sub_category,description,stemmed_tokens,similarity
0,Hornby 2014 Catalogue,Hobbies > Model Trains & Railway Sets > Rail V...,Product Description Hornby 2014 Catalogue Box ...,hornbi catalogu hobbi model train railway set ...,0.000000
1,FunkyBuys® Large Christmas Holiday Express Fes...,Hobbies > Model Trains & Railway Sets > Rail V...,Size Name:Large FunkyBuys® Large Christmas Hol...,funkybuy larg christma holiday express festiv ...,0.000000
2,CLASSIC TOY TRAIN SET TRACK CARRIAGES LIGHT EN...,Hobbies > Model Trains & Railway Sets > Rail V...,BIG CLASSIC TOY TRAIN SET TRACK CARRIAGE LIGHT...,classic toy train set track carriag light engi...,0.015509
3,HORNBY Coach R4410A BR Hawksworth Corridor 3rd,Hobbies > Model Trains & Railway Sets > Rail V...,Hornby 00 Gauge BR Hawksworth 3rd Class W 2107...,hornbi coach ra br hawksworth corridor rd hobb...,0.000000
4,Hornby 00 Gauge 0-4-0 Gildenlow Salt Co. Steam...,Hobbies > Model Trains & Railway Sets > Rail V...,Product Description Hornby RailRoad 0-4-0 Gild...,hornbi gaug gildenlow salt co steam locomot mo...,0.000000


# Define a function that returns the cosine similarity between two documents (in this case, product titles and descriptions).

TfidfVectorizer is a text feature extraction tool that creates a numerical representation of text by converting it into a matrix of TF-IDF (Term Frequency-Inverse Document Frequency) features. This matrix can then be used as input for various machine learning algorithms.

The tokenizer parameter in TfidfVectorizer is set to tokenize_stem, which is a custom function defined elsewhere in the code that tokenizes and stems text.

cosine_similarity is a function in scikit-learn that calculates the cosine similarity between two matrices. In this code, it is used to calculate the cosine similarity between two texts represented as TF-IDF matrices generated by TfidfVectorizer.

The similarity function defined in the code takes in two text inputs (text1 and text2), uses TfidfVectorizer to generate TF-IDF matrices for them, and then calculates the cosine similarity between these matrices using cosine_similarity. It returns the cosine similarity score as a single value.

# what is cosine similarity
Cosine similarity is a measure of similarity between two non-zero vectors of an inner product space that measures the cosine of the angle between them. In the context of natural language processing, it is commonly used to measure the similarity between two text documents. The cosine similarity score ranges from 0 to 1, where 0 means no similarity and 1 means identical. The score is calculated based on the frequency of common words or terms in the two documents. The higher the cosine similarity score, the more similar the documents are.


In [65]:
tfidf = TfidfVectorizer()

def similarity(text1, text2):
  matrix = tfidf.fit_transform([text1, text2])
  return cosine_similarity(matrix)[0][1]

# Define a function that takes a product title as string and returns a DataFrame with the top 10 most relevant products based on the cosine similarity between the query string and the product titles and descriptions.
The tokenize_stem function is called on the product string, which presumably tokenizes and stems the words in the string.
The similarity function is used to compute the cosine similarity between the stemmed query and each product's stemmed_tokens column. This function likely calculates the cosine similarity between two vectors.
A new column called similarity is added to the amzon_df dataframe using the apply method. This column contains the cosine similarity between the query and each product's stemmed_tokens column.
The amzon_df dataframe is sorted in descending order by the similarity column using the sort_values method.
The top 10 rows of the sorted dataframe are selected using the head method and the [['product_name', 'amazon_category_and_sub_category', 'description', 'similarity']] notation is used to return only the specified columns.

In [74]:
def recommend(product_name):
  stemmed_name = tokenize_stem(product_name)

  amazon_df['similarity'] = amazon_df['stemmed_tokens'].apply(lambda x: similarity(x, stemmed_name))

  result = amazon_df.sort_values(by='similarity', ascending=False).head(10)

    # Select relevant columns, including similarity
  return result[['product_name', 'amazon_category_and_sub_category', 'description', 'similarity']]

In [75]:
amazon_df['product_name'][5]

'20pcs Model Garden Light Double Heads Lamppost Scale 1:100'

In [78]:
recommend('Marvel Select Ghost Rider Action Figure')

,product_name,amazon_category_and_sub_category,description,similarity
1000,Marvel Select Ghost Rider Action Figure,Figures & Playsets > Science Fiction & Fantasy,Marvel Select Ghost Rider Action Figure,0.873669
6395,Marvel Select Silver Surfer Action Figure,Figures & Playsets > Science Fiction & Fantasy,Product Description Write Review Marvel Select...,0.392556
4021,Ghost Rider,Figures & Playsets > Knights & Castles,Ghost Rider Hand painted figure. Please note :...,0.308508
9175,"Marvel Universe 3 3/4"" Action Figures - Iron P...",Characters & Brands > Hasbro,Iron Patriot Marvel Universe Series 2 #19 Acti...,0.304126
6506,Marvel Select - Anti-Venom Special Collector E...,Figures & Playsets > Science Fiction & Fantasy,A Diamond Select Release! A Jean St Jean Sculp...,0.299471
7133,Marvel Super Hero Mashers Marvel's Spider-Man ...,Characters & Brands > Hasbro,Marvel Superhero Mashers action figure of Spid...,0.274548
9152,Transfixed Bilbo action figure lord of the rin...,Figures & Playsets > Science Fiction & Fantasy,Action figure,0.269637
7116,Marvel Comics Select Carnage Action Figure,Figures & Playsets > Science Fiction & Fantasy,Product Description A Diamond Select Toys rele...,0.268879
7081,Monsters vs Aliens Deluxe Action Figure Dr Coc...,Figures & Playsets > Science Fiction & Fantasy,Box Contains 1 x 6 inch The Missing Link actio...,0.260556
9573,Marvel Comics Select Zombie Sabretooth Action ...,Hobbies > Collectible Figures & Memorabilia > ...,Product Description A Diamond Select Toys rele...,0.259458
